# 01 - From saved reels to a catalogue 🎬

In the workshop we started from text and trained models on it.

Here we start from **your own data**: 91 movie reels you saved on Facebook and Instagram.

A RAG system is only as good as what it retrieves, and it can only retrieve
what you indexed. So step one is boring and decisive:

**messy bookmarks → clean, enriched records**

```
saved reels ──group──▶ distinct titles ──TMDb──▶ plot, runtime, genres, cast, streaming
                                        ──IMDb dump──▶ ratings
```

## 0. Setup 🧰

Run once in a terminal, in the `rag` folder (see README for the details):

```
uv sync --extra dev
```

Then pick the project's environment as the notebook kernel.

In [1]:
# Make the project importable even if it wasn't installed (uv sync installs it)
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

%load_ext autoreload
%autoreload 2

from movierag.config import get_config
cfg = get_config()
print(cfg.summary())

bookmarks     : /home/jp/llms/rag/movies-reels.json
state dir     : /home/jp/.movierag
LLM           : hf:Qwen/Qwen3-4B-Instruct-2507 (4-bit)
embeddings    : BAAI/bge-small-en-v1.5
reranker      : cross-encoder/ms-marco-MiniLM-L6-v2
TMDb key      : yes
YouTube key   : MISSING
watch region  : CH
offline mode  : False


## 1. Meet your bookmarks 📌

`movies-reels.json` was extracted from your saved collections: one record per reel,
with the caption and a best guess of which film it shows.

Look before you index. Same rule as fine-tuning: **never use data you haven't inspected.**

In [3]:
import pandas as pd
from movierag.catalogue import load_bookmarks

bookmarks = load_bookmarks(cfg.bookmarks_path)
df = pd.DataFrame([b.model_dump() for b in bookmarks])
print(len(df), "reels")
df[["id", "platform", "title", "year", "kind", "match_status", "caption"]].head(10)

91 reels


,id,platform,title,year,kind,match_status,caption
0,FB001,Facebook,The Looking Planet,2014.0,Short,Confirmed,The Looking Planet
1,FB002,Facebook,Pipeline,2021.0,Film,Confirmed,Ce film
2,FB003,Facebook,Devs,2020.0,TV mini series,Confirmed,Questions whether free will remains meaningful...
3,FB004,Facebook,A Very Long Engagement (Un long dimanche de fi...,2004.0,Film,Confirmed,Des soldats essaient de rentré chez eux #resum...
4,FB005,Facebook,Siegfried,2005.0,Film,Confirmed,Le Jeune Homme Sauvage Qui Pouvait Parler aux ...
5,FB006,Facebook,Oruzhie (The Weapon),2012.0,TV movie,Candidate,🎥🎬🍿
6,FB007,Facebook,The Count of Monte Cristo,2024.0,Film,Candidate,Fimz12's Video
7,FB008,Facebook,Mortal Kombat II,2026.0,Film,Confirmed,🔥 One of the BEST scenes in Mortal Kombat John...
8,FB009,Facebook,Doomsday,2008.0,Film,Confirmed,Doomsday
9,FB010,Facebook,Flight of Fury,2007.0,Video,Confirmed,Flight of fury #fblifestyle


In [ ]:
print(df["platform"].value_counts(), "\n")
print(df["match_status"].value_counts(), "\n")
print(df["kind"].value_counts())

platform
Facebook     85
Instagram     6
Name: count, dtype: int64 

match_status
Confirmed           80
Candidate            8
Unresolved           2
Confirmed (TMDb)     1
Name: count, dtype: int64 

kind
Film                       59
TV series                  15
TV movie                    4
TV episode                  4
Short                       2
Unidentified                2
TV mini series              1
Video                       1
Animated anthology film     1
Short film                  1
Anime TV series             1
Name: count, dtype: int64


### 🤔 What do you notice?

- Captions are a mix of French, English, Thai, Chinese... and hashtags.
- Some reels point to the **same** film (you saved *Kalki 2898 AD* twice).
- Some point to a single **episode** of a series.
- A couple are **unresolved**: nobody could tell what film it was.

The caption alone would make a poor search document. We need real metadata.

## 2. One title per film 🧮

We group the reels by IMDb ID. Several reels for one title is a useful signal by itself:
you were interested more than once.

In [5]:
from movierag.catalogue import group_bookmarks

groups, skipped = group_bookmarks(bookmarks)
print(f"{len(bookmarks)} reels -> {len(groups)} titles, {len(skipped)} unresolved")

for key, group in groups.items():
    if len(group) > 1:
        print(f"  {key}: {[b.title for b in group]}")

91 reels -> 87 titles, 2 unresolved
  imdb:tt12735488: ['Kalki 2898 AD', 'Kalki 2898 AD']
  imdb:tt0106950: ['Fortress', 'Fortress']


## 3. Ask TMDb about one film 🔎

IMDb has no free API. **TMDb** (The Movie Database) is the usual workaround and it can
look things up *by IMDb ID* with its `/find` endpoint.

Let's do one call by hand to see what comes back. (Calls are cached on disk:
re-running costs nothing.)

In [6]:
from movierag.tmdb import TMDbClient

tmdb = TMDbClient.from_config(cfg)
print("TMDb available:", tmdb.available)

hit = tmdb.find_by_imdb("tt8134186")   # Devs (2020)
print("find ->", hit)

TMDb available: True
find -> ('tv', 81349, None)


In [7]:
if hit:
    media_type, tmdb_id, _ = hit
    details = tmdb.details(media_type, tmdb_id)
    print("Top-level keys:", sorted(details)[:25], "...")
    devs = tmdb.to_title(media_type, details)
    print()
    print(devs.embedding_text())
else:
    print("No TMDb key (or offline mode): add TMDB_API_KEY to .env to see real data.")

Top-level keys: ['adult', 'aggregate_credits', 'backdrop_path', 'created_by', 'episode_run_time', 'external_ids', 'first_air_date', 'genres', 'homepage', 'id', 'in_production', 'keywords', 'languages', 'last_air_date', 'last_episode_to_air', 'name', 'networks', 'next_episode_to_air', 'number_of_episodes', 'number_of_seasons', 'origin_country', 'original_language', 'original_name', 'overview', 'popularity'] ...

Devs (2020) - series
Genres: Science Fiction, Fantasy, Drama, Mystery
Tagline: Reality is stranger than dreams.
Plot: Young computer engineer Lily Chan investigates the secretive development division of her employer, a cutting-edge tech company based in San Francisco, which she believes is behind the murder of her boyfriend.
Keywords: simulation, silicon valley, mysterious, complex, anxious, determinism, intimate, baffled, distressing
By: Alex Garland
Cast: Sonoya Mizuno, Nick Offerman, Jin Ha, Cailee Spaeny, Stephen McKinley Henderson


### 💡 `embedding_text()` is the document

That block of text is what we will **embed** (turn into a vector) and **index for keywords**.

Choosing what goes into it is one of the most important RAG decisions:
plot + genres + keywords + people + *why you saved it* (the reel caption).

## 4. Enrich everything 🏗️

For every title: `/find` by IMDb ID → `/movie/{id}` or `/tv/{id}` with keywords, credits
and streaming providers for Switzerland (`WATCH_REGION=CH`). Then IMDb ratings from IMDb's
official non-commercial dataset dump.

About 170 API calls the first time, under a minute. Instant afterwards.

In [8]:
from movierag.catalogue import build_catalogue, save_catalogue

titles = build_catalogue(bookmarks, tmdb, ratings_cache_dir=cfg.cache_dir)
save_catalogue(titles, cfg.catalogue_path)
print("saved to", cfg.catalogue_path)

91 reels -> 87 distinct titles (2 unresolved, skipped)
  enriched 10/87
  enriched 20/87
  enriched 30/87
  enriched 40/87
  enriched 50/87
  enriched 60/87
  enriched 70/87
  enriched 80/87
IMDb ratings attached to 84 titles
catalogue: 85 titles, 85 enriched from TMDb, TMDb network calls: 169
saved to /home/jp/.movierag/catalogue.json


In [10]:
cat = pd.DataFrame([{
    "title": t.title, "kind": t.kind, "year": t.year, "min": t.runtime_min,
    "genres": ", ".join(t.genres[:3]), "imdb": t.imdb_rating,
    "streaming (CH)": ", ".join(t.providers[:2]), "saved": len(t.bookmarks), "confidence": t.confidence,
} for t in titles])
cat.sort_values("imdb", ascending=False).head(20)

,title,kind,year,min,genres,imdb,streaming (CH),saved,confidence
36,Doctor Who,series,2005,60,"Action, Adventure, Drama",8.5,,3,confirmed
82,Requiem for a Dream,film,2000,102,"Crime, Drama",8.3,Canal+,1,confirmed
72,Bloodhounds,series,2023,61,"Drama, Action, Adventure",8.2,Netflix,1,candidate
79,Something the Lord Made,film,2004,110,Drama,8.0,,1,confirmed
55,WandaVision,series,2021,51,"Science Fiction, Fantasy, Mystery",7.9,Disney Plus,1,confirmed
54,Life of Pi,film,2012,127,"Adventure, Drama",7.9,Disney Plus,1,confirmed
52,Da Vinci's Demons,series,2013,60,Drama,7.9,,1,confirmed
80,Black Narcissus,film,1947,100,Drama,7.7,"ARD Plus Amazon channel , Artiflix",1,confirmed
3,A Very Long Engagement,film,2004,133,"Drama, Romance, War",7.6,,1,confirmed
2,Devs,series,2020,52,"Science Fiction, Fantasy, Drama",7.6,Disney Plus,1,confirmed


## 5. Check the gaps 🕳️

Anything that didn't get a plot will be hard to find later. Look at them now.

In [11]:
missing = [t for t in titles if not t.overview.strip()]
print(f"{len(missing)} titles without a plot:")
for t in missing:
    print(" -", t.title, t.year, t.item_id)

print()
print(cat["kind"].value_counts())

0 titles without a plot:

kind
film      63
series    19
short      3
Name: count, dtype: int64


# ✅ What we just built

- 91 noisy reels became a **catalogue** of distinct, enriched titles
- each title has a **document** (`embedding_text`) and **structured fields** (year, runtime, kind, genres)
- everything is cached and saved in `catalogue.json`, the single source of truth

Notice that we have two kinds of information:

| Kind | Example | Best handled by |
|---|---|---|
| structured | runtime = 104 min | **filters** |
| unstructured | "a lonely alien in Scotland" | **search** |

## Next: turn documents into vectors and search them 🧭